# Практика · Що бачить мережа> Лекція: [lecture.html](lecture.html) · Тест: [quiz.html](quiz.html) · ДЗ: [homework.html](homework.html)> ⏱ **Зошит навчає дві мережі й робить понад пʼять тисяч прогонів заслону.**> Заміряно `check_notebook.py` двома прогонами: **82 і 91 секунда** на чотириядерному процесорі> **в один потік**, без відеокарти. Найдовші місця — два навчання (по 20-30 с> кожне); решта майже миттєва. Остання клітинка друкує фактичний час твого прогону.Що зробимо:1. навчимо маленьку CNN на фігурах 28×28 шести класів;2. **замір 1** — порівняємо ядра першого шару з Собелем, і не на око, а косинусом,   з орієнтиром на 200 випадкових ядер;3. **замір 2** — знімемо карти активацій усіх трьох шарів і поміряємо розрідження;4. напишемо **власну реалізацію Grad-CAM** без жодних сторонніх бібліотек;5. звіримо ваги карти, **пораховані руками**, з автоматичними — двічі, через   `np.allclose`;6. **замір 3** — побудуємо карти для шести зображень і для шести класів;7. **замір 4** — побудуємо карту заслону й порахуємо, скільки вона коштує;8. **замір 5** — поміряємо, чи згодні два методи між собою;9. **замір 6** — ⚠️ головне: зіпсуємо мережі ваги й подивимось, чи змінилась карта;10. **замір 7** — покажемо випадок, коли карта бездоганна, а відповідь хибна.

## 1 · Середовище`torch.set_num_threads(1)` — перший рядок, і не для краси. На маленьких тензорахкілька потоків більше домовляються між собою, ніж рахують. Але важливіше інше:під кількома потоками числа з рухомою комою додаються в іншому порядку, і сумипливуть від прогону до прогону. Нам потрібні числа, які збігаються з лекцієюдо останнього знака.

In [ ]:
import copy
import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

# один потік: і швидше на дрібних тензорах, і числа не пливуть від прогону до прогону
torch.set_num_threads(1)

notebook_started = time.perf_counter()

print("torch      :", torch.__version__)
print("numpy      :", np.__version__)
print("потоків CPU:", torch.get_num_threads())

## 2 · Датасет: шість класів фігур 28×28Той самий генератор, що в усьому блоці. Нічого не завантажується: кожна фігура —це формула плюс гаусів шум. Шість класів означає, що **вгадування навмання дає0.167** — це наш нижній орієнтир на весь зошит.

In [ ]:
SHAPE_NAMES = ["коло", "квадрат", "ромб", "кільце", "хрест", "трикутник"]


def draw_shape(kind, rng, size=28, jitter=4, noise=0.20, center=None, radius=None):
    """Малює одну фігуру заданого класу як масив 28×28 зі значеннями 0..1."""
    image = np.zeros((size, size), dtype=np.float32)
    # центр зсуваємо, щоб мережа не завчила одне-єдине положення предмета
    if center is None:
        center_y = size / 2 + rng.integers(-jitter, jitter + 1)
        center_x = size / 2 + rng.integers(-jitter, jitter + 1)
    else:
        center_y, center_x = center
    if radius is None:
        radius = rng.integers(5, 9)

    # відстані кожного пікселя від центра — з них складаються всі шість фігур
    yy, xx = np.mgrid[0:size, 0:size]
    dy, dx = yy - center_y, xx - center_x

    if kind == 0:                                    # коло
        image[dy * dy + dx * dx <= radius * radius] = 1.0
    elif kind == 1:                                  # квадрат
        image[(np.abs(dy) <= radius * 0.85) & (np.abs(dx) <= radius * 0.85)] = 1.0
    elif kind == 2:                                  # ромб
        image[np.abs(dy) + np.abs(dx) <= radius] = 1.0
    elif kind == 3:                                  # кільце
        distance = dy * dy + dx * dx
        image[(distance <= radius * radius) & (distance >= (radius - 3) ** 2)] = 1.0
    elif kind == 4:                                  # хрест
        image[(np.abs(dy) <= 2) & (np.abs(dx) <= radius)] = 1.0
        image[(np.abs(dx) <= 2) & (np.abs(dy) <= radius)] = 1.0
    else:                                            # трикутник
        image[(dy >= -radius * 0.8) & (dy <= radius * 0.8)
              & (np.abs(dx) <= (dy + radius * 0.8) * 0.6)] = 1.0

    image += rng.normal(0, noise, image.shape).astype(np.float32)
    return np.clip(image, 0, 1)


def make_dataset(count, rng):
    """Повертає (count, 1, 28, 28) і (count,). Класи чергуються, тож їх порівну."""
    images = np.zeros((count, 1, 28, 28), dtype=np.float32)
    labels = np.zeros(count, dtype=np.int64)
    for i in range(count):
        slot = i % len(SHAPE_NAMES)
        images[i, 0] = draw_shape(slot, rng)
        labels[i] = slot
    return torch.from_numpy(images), torch.from_numpy(labels)


rng = np.random.default_rng(42)
train_x, train_y = make_dataset(1800, rng)
test_x, test_y = make_dataset(600, rng)

print(f"навчальних прикладів: {len(train_x)}, перевірочних: {len(test_x)}")
print(f"форма батча: {tuple(train_x.shape)}, класів: {len(SHAPE_NAMES)}")
print(f"рівень вгадування: {1 / len(SHAPE_NAMES):.3f}")

## 3 · МережаТри згорткові шари, глобальне усереднення, один лінійний шар. Зверни увагу на`self.relu3` — це **окремий модуль**, а не виклик `F.relu`. Так зроблено навмисно:саме на цей модуль ми повісимо перехоплювач Grad-CAM. Класичний Grad-CAM берекарти ознак **після** нелінійності, а якщо чіпляти hook на `conv3`, дістанеш вихіддо ReLU — інші числа й інший метод.

In [ ]:
class Net(nn.Module):
    """Маленька CNN: 8 → 16 → 32 канали, глобальне усереднення, лінійна голова."""

    def __init__(self, n_classes=6):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 8, 3, padding=1)
        self.conv2 = nn.Conv2d(8, 16, 3, padding=1)
        self.conv3 = nn.Conv2d(16, 32, 3, padding=1)
        self.relu1 = nn.ReLU()
        self.relu2 = nn.ReLU()
        self.relu3 = nn.ReLU()      # ← саме сюди чіплятимемо hook Grad-CAM
        self.pool = nn.MaxPool2d(2)
        self.fc = nn.Linear(32, n_classes)

    def features(self, x):
        """Повертає всі проміжні карти — вони знадобляться в замірі 2."""
        a1 = self.relu1(self.conv1(x))       # 8 × 28 × 28
        p1 = self.pool(a1)                   # 8 × 14 × 14
        a2 = self.relu2(self.conv2(p1))      # 16 × 14 × 14
        p2 = self.pool(a2)                   # 16 × 7 × 7
        a3 = self.relu3(self.conv3(p2))      # 32 × 7 × 7
        return a1, p1, a2, p2, a3

    def forward(self, x):
        a3 = self.features(x)[-1]
        return self.fc(a3.mean(dim=(2, 3)))  # глобальне усереднення → 32 числа


def count_params(module):
    return sum(p.numel() for p in module.parameters())


torch.manual_seed(0)
model = Net()

# рахуємо руками: кожна згортка має cin·cout·3·3 ваг плюс cout зсувів
conv1_by_hand = 1 * 8 * 3 * 3 + 8
conv2_by_hand = 8 * 16 * 3 * 3 + 16
conv3_by_hand = 16 * 32 * 3 * 3 + 32
fc_by_hand = 32 * 6 + 6
total_by_hand = conv1_by_hand + conv2_by_hand + conv3_by_hand + fc_by_hand

print(f"conv1 руками: {conv1_by_hand:5d}   бібліотека: {count_params(model.conv1):5d}")
print(f"conv2 руками: {conv2_by_hand:5d}   бібліотека: {count_params(model.conv2):5d}")
print(f"conv3 руками: {conv3_by_hand:5d}   бібліотека: {count_params(model.conv3):5d}")
print(f"fc    руками: {fc_by_hand:5d}   бібліотека: {count_params(model.fc):5d}")
print(f"разом руками: {total_by_hand:5d}   бібліотека: {count_params(model):5d}")

assert total_by_hand == count_params(model), "розрахунок розійшовся!"
print("\n✅ збігається")

## 4 · Навчання30 епох на 1800 прикладах. Це перше з двох навчань у зошиті — приблизно 30 секунд.

In [ ]:
def train(model, train_x, train_y, epochs=30, lr=3e-3, batch=64, seed=0):
    """Звичайний Adam. seed фіксує порядок перемішування, щоб числа повторювались."""
    torch.manual_seed(seed)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    n = len(train_x)
    for _ in range(epochs):
        order = torch.randperm(n)
        for start in range(0, n, batch):
            index = order[start:start + batch]
            optimizer.zero_grad()
            loss_fn(model(train_x[index]), train_y[index]).backward()
            optimizer.step()
    return model


def accuracy(model, x, y):
    model.eval()
    with torch.no_grad():
        return (model(x).argmax(1) == y).float().mean().item()


started = time.perf_counter()
train(model, train_x, train_y)
train_seconds = time.perf_counter() - started
test_accuracy = accuracy(model, test_x, test_y)

print(f"навчання зайняло {train_seconds:.1f} с")
print(f"точність на перевірці: {test_accuracy:.3f} (вгадування дало б 0.167)")

## 5 · ЗАМІР 1 · Чи стали ядра першого шару детекторами межЯдра першого шару — єдині, які можна намалювати як картинку: у них один вхіднийканал, тобто девʼять чисел на ядро. Порівняємо їх із класичними ядрами з теми 04**косинусною подібністю** — числом від −1 до +1, яке не зважає на масштаб.І одразу зробимо те, без чого це порівняння нічого не варте: візьмемо **орієнтир**із 200 цілком випадкових ядер. Без нього неможливо сказати, чи 0.6 — це багато.

In [ ]:
def cos_sim(a, b):
    """Косинус між двома ядрами, розкладеними в рядки. Масштаб не має значення."""
    a = np.asarray(a, dtype=np.float64).ravel()
    b = np.asarray(b, dtype=np.float64).ravel()
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-12))


# перевіряємо саму функцію на прикладі, який можна порахувати в голові:
# [1, 0, -1] і [2, 0, -2] — те саме ядро, лише вдвічі яскравіше
print("cos([1,0,-1], [2,0,-2]) =", round(cos_sim([1, 0, -1], [2, 0, -2]), 6))
assert abs(cos_sim([1, 0, -1], [2, 0, -2]) - 1.0) < 1e-9
print("✅ функція поводиться як треба\n")

sobel_x = np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=np.float64)
sobel_y = sobel_x.T.copy()
laplace = np.array([[0, 1, 0], [1, -4, 1], [0, 1, 0]], dtype=np.float64)
uniform = np.ones((3, 3)) / 9.0

trained_kernels = model.conv1.weight.detach().numpy()[:, 0]      # 8 × 3 × 3

print("ядро |  Собель-X |  Собель-Y | лапласіан | рівномірне | сума ваг")
for i, kernel in enumerate(trained_kernels):
    print(f"{i:4d} | {cos_sim(kernel, sobel_x):9.3f} | {cos_sim(kernel, sobel_y):9.3f} | "
          f"{cos_sim(kernel, laplace):9.3f} | {cos_sim(kernel, uniform):10.3f} | "
          f"{kernel.sum():8.2f}")

Тепер орієнтири. Три числа, з якими треба порівнювати таблицю вище:- найкращий збіг **навчених** ядер із Собелем;- найкращий збіг **випадкових** ядер із Собелем — стеля, яку дає чистий шум;- найкращий збіг **тих самих восьми ядер до навчання** — те, з чого мережа почала.

In [ ]:
def best_edge_match(kernels):
    """Найбільший за модулем збіг із будь-яким із двох ядер Собеля."""
    return max(max(abs(cos_sim(k, sobel_x)), abs(cos_sim(k, sobel_y))) for k in kernels)


# 200 випадкових ядер — щоб знати, скільки «схожості» дає чистий шум
torch.manual_seed(123)
random_kernels = torch.randn(200, 3, 3).numpy()

# ті самі вісім ядер до навчання: manual_seed(0) відтворює початкову ініціалізацію
torch.manual_seed(0)
fresh_kernels = Net().conv1.weight.detach().numpy()[:, 0]

random_edge_scores = [max(abs(cos_sim(k, sobel_x)), abs(cos_sim(k, sobel_y)))
                      for k in random_kernels]

print(f"навчені ядра, найкращий збіг із Собелем : {best_edge_match(trained_kernels):.3f}")
print(f"ті самі ядра ДО навчання                : {best_edge_match(fresh_kernels):.3f}")
print(f"200 випадкових ядер, найкращий          : {max(random_edge_scores):.3f}")
print(f"200 випадкових ядер, середній           : {np.mean(random_edge_scores):.3f}")

print(f"\nнайкращий збіг із РІВНОМІРНИМ ядром: "
      f"{max(abs(cos_sim(k, uniform)) for k in trained_kernels):.3f} після навчання, "
      f"{max(abs(cos_sim(k, uniform)) for k in fresh_kernels):.3f} до")

big_after = int((np.abs(trained_kernels.sum(axis=(1, 2))) > 1.5).sum())
big_before = int((np.abs(fresh_kernels.sum(axis=(1, 2))) > 1.5).sum())
print(f"ядер із |сумою ваг| > 1.5: до навчання {big_before} з 8, після {big_after} з 8")

**Що з цього виходить.** Найкраще навчене ядро схоже на Собеля слабше, ніж найкращез двохсот випадкових. Навчання зрушило це число на соті долі — тобто ні на що.**Детекторів меж на нашому датасеті не зʼявилось**, і це чесний результат: нашіфігури суцільні, всередині них нічого немає, детектувати нема чого.Зате зʼявилось інше. Три ядра з восьми стали майже рівномірними, з великою додатноюсумою ваг — це **детектори заповненості**, «скільки тут загалом світлого». Донавчання таких не було жодного.

## 6 · Зразкові зображенняДалі всі заміри робимо на шести центрованих фігурах — по одній на клас. Центруємонавмисно: інакше не зрозуміло, карта зсунулась через мережу чи через сам предмет.

In [ ]:
probe_rng = np.random.default_rng(7)
probe_images = np.stack([draw_shape(kind, probe_rng, center=(14, 14), radius=7)
                         for kind in range(6)])
probe_batch = torch.from_numpy(probe_images[:, None].astype(np.float32))

model.eval()
with torch.no_grad():
    probe_probs = torch.softmax(model(probe_batch), 1)
probe_pred = probe_probs.argmax(1).numpy()

print("що на вході | відповідь мережі | упевненість")
for i in range(6):
    mark = "✅" if probe_pred[i] == i else "❌"
    print(f"{SHAPE_NAMES[i]:11s} | {SHAPE_NAMES[probe_pred[i]]:16s} | "
          f"{probe_probs[i, probe_pred[i]].item():.3f} {mark}")

## 7 · ЗАМІР 2 · Карти активацій по шарахОдин вхід, три шари. Дивимось на чотири числа:- **частка ненульових** — скільки клітинок узагалі щось передали далі;- **максимум** — наскільки сильний найгучніший відгук;- **маса в 10 % клітинок** — наскільки активація зібрана в кількох точках;- **мертві карти** — скільки карт на цьому вході мовчать цілком.Підручник обіцяє, що від шару до шару відгук стає розрідженішим. Подивись, що вийде.

In [ ]:
def layer_stats(activation):
    """Чотири числа про одну карту активацій: (n, h, w) без розміру батча."""
    values = activation.numpy()
    sorted_values = np.sort(values.ravel())[::-1]
    top_tenth = max(1, int(len(sorted_values) * 0.10))
    return {
        "nonzero": float((values > 1e-6).mean()),
        "max": float(values.max()),
        # яка частка всієї активації припадає на 10 % найяскравіших клітинок
        "concentration": float(sorted_values[:top_tenth].sum() / (sorted_values.sum() + 1e-12)),
        "dead": int((values.max(axis=(1, 2)) <= 1e-6).sum()),
        "count": values.shape[0],
    }


with torch.no_grad():
    a1, p1, a2, p2, a3 = model.features(probe_batch[0:1])

print("вхід — «коло»\n")
print("шар          | форма        | ненульових | максимум | маса в 10 % | мертвих карт")
for name, act in [("conv1 + ReLU", a1[0]), ("conv2 + ReLU", a2[0]), ("conv3 + ReLU", a3[0])]:
    s = layer_stats(act)
    shape = f"{s['count']} × {act.shape[1]} × {act.shape[2]}"
    print(f"{name:12s} | {shape:12s} | {s['nonzero']:10.3f} | {s['max']:8.2f} | "
          f"{s['concentration']:11.3f} | {s['dead']:5d} з {s['count']}")

Перевіримо, чи це не випадковість одного зображення — прогонимо ще два входи.

In [ ]:
for image_index in (4, 5):
    with torch.no_grad():
        b1, _, b2, _, b3 = model.features(probe_batch[image_index:image_index + 1])
    fractions = [layer_stats(t[0])["nonzero"] for t in (b1, b2, b3)]
    dead = [layer_stats(t[0])["dead"] for t in (b1, b2, b3)]
    print(f"{SHAPE_NAMES[image_index]:10s}: ненульових "
          f"{fractions[0]:.3f} → {fractions[1]:.3f} → {fractions[2]:.3f}   "
          f"мертвих карт {dead[0]} → {dead[1]} → {dead[2]}")

**Розрідження не монотонне.** Другий шар справді тихіший за перший, а третій зновурозговорився — і так на всіх трьох входах. Підручникова картинка на нашій мережіне відтворилась, і замовчувати це не можна.Монотонне інше: **мертвих карт стає більше** — 0, потім 4 з 16, потім 9 з 32. Осьце і є спеціалізація: у глибшому шарі більшість карт налаштована на речі, яких нацьому зображенні немає.

## 8 · Власна реалізація Grad-CAMЖодних бібліотек. Уся механіка — чотири дії:1. `register_forward_hook` перехоплює вихід потрібного шару, коли мережа йде вперед;2. `retain_grad()` просить PyTorch зберегти градієнт цього проміжного тензора   (за замовчуванням зберігаються градієнти лише ваг);3. `backward()` від **логіта одного класу** — не від помилки, як при навчанні;4. ваги = середнє градієнта по простору, карта = `relu` зваженої суми.

In [ ]:
def grad_cam(model, image, class_index, layer_name="relu3"):
    """Grad-CAM своїми руками. Повертає (карта, активації, градієнти, ваги)."""
    model.eval()
    stored = {}

    def hook(module, inputs, output):
        # без retain_grad градієнт проміжного тензора PyTorch викине одразу після backward
        output.retain_grad()
        stored["activation"] = output

    handle = getattr(model, layer_name).register_forward_hook(hook)
    model.zero_grad()
    logits = model(image.unsqueeze(0))
    logits[0, class_index].backward()      # ← зворотний прохід від ОДНОГО числа
    handle.remove()

    activation = stored["activation"][0]            # 32 × 7 × 7
    gradient = stored["activation"].grad[0]         # 32 × 7 × 7
    weights = gradient.mean(dim=(1, 2))             # 32 — по одному числу на карту
    cam = torch.relu((weights[:, None, None] * activation).sum(0))
    return (cam.detach().numpy(), activation.detach().numpy(),
            gradient.detach().numpy(), weights.detach().numpy())


cam_circle, act_circle, grad_circle, weights_circle = grad_cam(
    model, probe_batch[0], int(probe_pred[0]))

print(f"карта має розмір {cam_circle.shape} на вході 28×28")
print(f"→ одна клітинка карти = квадрат {28 // cam_circle.shape[0]}×{28 // cam_circle.shape[0]} пікселів")
print(f"ваги w: від {weights_circle.min():+.4f} до {weights_circle.max():+.4f}, "
      f"додатних {int((weights_circle > 0).sum())} з 32")
print(f"максимум карти: {cam_circle.max():.3f}")

## 9 · Ручний підрахунок ваг проти автоматичногоДві звірки, і друга цікавіша за першу.**Перша.** Ваги — це просто середнє градієнта по 49 клітинках. Порахуємо це`numpy`-ем окремо й порівняємо з тим, що дав `torch`.**Друга.** У нашій мережі за останнім згортковим шаром стоїть глобальне усередненняй один лінійний шар. Логіт класу *c* дорівнює сумі `fc[c][k]`, помножених на середнєкарти *k*. Похідна такого виразу по будь-якій клітинці карти *k* дорівнює`fc[c][k] / 49` і **однакова в усіх 49 клітинках**. Отже, ваги Grad-CAM мусятьдорівнювати рядку матриці `fc`, поділеному на 49 — точно, без наближень.

In [ ]:
# звірка 1: середнє градієнта, пораховане руками
weights_by_hand = grad_circle.mean(axis=(1, 2))
assert np.allclose(weights_by_hand, weights_circle, atol=1e-7), "ваги розійшлись!"
print("✅ ваги руками == g.mean(dim=(1,2))")
print(f"   максимальна різниця: {np.abs(weights_by_hand - weights_circle).max():.2e}")

# звірка 2: ваги мусять дорівнювати рядку fc, поділеному на 49
fc_row = model.fc.weight[int(probe_pred[0])].detach().numpy() / 49.0
assert np.allclose(fc_row, weights_circle, atol=1e-8), "тотожність не справдилась!"
print("\n✅ ваги == рядок fc / 49")
print(f"   максимальна різниця: {np.abs(fc_row - weights_circle).max():.2e}")

# і найважливіший наслідок: градієнт усередині карти сталий
inside_spread = grad_circle.std(axis=(1, 2)).max()
print(f"\nрозкид градієнта ВСЕРЕДИНІ карти: {inside_spread:.2e} — тобто нуль")
print("Наслідок: у такій мережі ваги Grad-CAM НЕ ЗАЛЕЖАТЬ від зображення взагалі.")
print("Вони фіксовані для класу. Від картинки залежать лише самі карти ознак.")

Ще одне число, яке варто мати: скільки клітинок викидає `relu` на останньому кроці.І окремо — чи повʼязана яскравість карти ознак із її вагою. Спокуса «найяскравішакарта і є головна» коштує дорого, тож перевіримо її числом.

In [ ]:
weighted_sum = (weights_circle[:, None, None] * act_circle).sum(0)
zeroed = float((weighted_sum < 0).mean())
print(f"relu обнуляє {zeroed:.3f} клітинок, тобто {int(zeroed * 49)} із 49")

channel_brightness = act_circle.reshape(32, -1).mean(1)
correlation = np.corrcoef(channel_brightness, weights_circle)[0, 1]
print(f"\nкореляція «середня яскравість карти» ↔ «вага в Grad-CAM»: {correlation:+.3f}")
print("Тобто яскравість майже нічого не каже про важливість.")

## 10 · ЗАМІР 3 · Карти для шести зображень і для шести класівСпершу — карта правильного класу для кожної фігури. Потім найцікавіше: **однезображення, шість запитань**. Grad-CAM будується для класу, а не для зображення,тож на тому самому колі можна спитати «а де докази за квадрат?».

In [ ]:
cams = {}
for i in range(6):
    cams[i], _, _, _ = grad_cam(model, probe_batch[i], int(probe_pred[i]))
    nonzero_cells = int((cams[i] > 1e-6).sum())
    print(f"{SHAPE_NAMES[i]:10s}: максимум {cams[i].max():.3f}, "
          f"ненульових клітинок {nonzero_cells}/49")

print(f"\nрозкид максимумів: від {min(c.max() for c in cams.values()):.3f} "
      f"до {max(c.max() for c in cams.values()):.3f} — у "
      f"{max(c.max() for c in cams.values()) / min(c.max() for c in cams.values()):.1f} раза.")
print("Після нормалізації перед показом усі шість виглядатимуть однаково яскраво.")

In [ ]:
def upsample(matrix, size=28):
    """Розтягує карту n×n до size×size білінійно — так само, як роблять перед показом."""
    tensor = torch.from_numpy(np.asarray(matrix, dtype=np.float32))[None, None]
    return F.interpolate(tensor, size=(size, size), mode="bilinear",
                         align_corners=False)[0, 0].numpy()


def pearson(a, b):
    """Кореляція двох карт. NaN, якщо в котроїсь немає розкиду (порожня карта)."""
    a = np.asarray(a, dtype=np.float64).ravel()
    b = np.asarray(b, dtype=np.float64).ravel()
    if a.std() < 1e-12 or b.std() < 1e-12:
        return float("nan")
    return float(np.corrcoef(a, b)[0, 1])


print("Одне зображення «коло», шість різних запитань до мережі:\n")
print("питаємо про клас | упевненість | максимум карти | кореляція з картою «коло»")
maps_by_class = {}
for c in range(6):
    maps_by_class[c], _, _, _ = grad_cam(model, probe_batch[0], c)
    r = pearson(upsample(maps_by_class[0]), upsample(maps_by_class[c]))
    print(f"{SHAPE_NAMES[c]:16s} | {probe_probs[0, c].item():11.3f} | "
          f"{maps_by_class[c].max():14.3f} | {r:+.3f}")

Карта класу «квадрат» майже збігається з картою класу «коло», хоча мережа дає«квадрату» майже нульову впевненість. Причина пряма: ваги дорівнюють рядкамматриці `fc`, і якщо рядки для двох класів схожі, то й карти будуть схожі.**«Карта для класу c» іноді показує не те, що відрізняє клас c, а те, що спільнедля цілої групи класів.**

## 11 · Роздільність: conv3 проти conv2Карта з останнього шару має розмір 7×7 — одна клітинка на 16 пікселів входу.Ранішній шар дасть 14×14. Порахуємо, наскільки дві карти відрізняються.

In [ ]:
cam_conv2, _, _, _ = grad_cam(model, probe_batch[0], int(probe_pred[0]), layer_name="relu2")

print(f"conv3 (останній шар): карта {cams[0].shape} → одна клітинка {28 // 7}×{28 // 7} px")
print(f"conv2 (ранішній шар) : карта {cam_conv2.shape} → одна клітинка {28 // 14}×{28 // 14} px")
print(f"\nкореляція між картами двох шарів: "
      f"{pearson(upsample(cams[0]), upsample(cam_conv2)):+.3f}")

## 12 · ЗАМІР 4 · Чутливість до заслонуЗовсім інший метод: жодних градієнтів, мережа лишається чорною скринькою. Рухаємопо зображенню сірий квадрат і дивимось, наскільки падає впевненість у правильномукласі.Ціна методу — головне, на що тут треба дивитись.

In [ ]:
def occlusion_map(model, image, class_index, patch=7, stride=1, fill=0.5):
    """Карта падіння впевненості. Повертає (карта, базова впевненість, прогонів, позиції)."""
    model.eval()
    with torch.no_grad():
        base = torch.softmax(model(image.unsqueeze(0)), 1)[0, class_index].item()

    positions = list(range(0, 28 - patch + 1, stride))
    occluded_batch = []
    for row in positions:
        for col in positions:
            occluded = image.clone()
            occluded[0, row:row + patch, col:col + patch] = fill
            occluded_batch.append(occluded)

    with torch.no_grad():
        probs = torch.softmax(model(torch.stack(occluded_batch)), 1)[:, class_index].numpy()

    drops = (base - probs).reshape(len(positions), len(positions))
    return drops, base, len(occluded_batch), positions


print("квадрат | крок | прогонів | макс. падіння | мін. падіння")
for patch in (5, 7, 9, 11):
    drops, base, runs, _ = occlusion_map(model, probe_batch[0], int(probe_pred[0]), patch, 1)
    print(f"{patch:2d}×{patch:<2d}   |    1 | {runs:8d} | {drops.max():13.3f} | {drops.min():12.3f}")

print(f"\nбазова впевненість мережі: {base:.3f}")
print(f"Grad-CAM коштує 2 прогони. Заслін 7×7 із кроком 1 — 484, тобто в 242 рази більше.")
print("Мінімальне падіння відʼємне: подекуди заслін ПІДВИЩУЄ впевненість мережі.")

## 13 · ЗАМІР 5 · Чи згодні два методи між собоюОсь навіщо потрібен був другий метод. Grad-CAM читає градієнти всередині мережі,заслін туди не заглядає взагалі. Якщо обидва показують на те саме — гіпотезапідкріплена. Якщо ні — довіряти не можна жодному.

In [ ]:
def occlusion_to_pixels(drops, positions, patch):
    """Розкладає падіння назад по пікселях: кожен піксель — середнє по вікнах, що його вкрили."""
    total = np.zeros((28, 28), dtype=np.float64)
    count = np.zeros((28, 28), dtype=np.float64)
    for i, row in enumerate(positions):
        for j, col in enumerate(positions):
            total[row:row + patch, col:col + patch] += drops[i, j]
            count[row:row + patch, col:col + patch] += 1
    count[count == 0] = 1
    return total / count


occlusion_pixel_maps = {}
agreement = []
print("зображення | упевненість | кореляція Grad-CAM ↔ заслін")
for i in range(6):
    drops, base, runs, positions = occlusion_map(model, probe_batch[i], int(probe_pred[i]), 7, 1)
    occlusion_pixel_maps[i] = occlusion_to_pixels(drops, positions, 7)
    r = pearson(upsample(cams[i]), occlusion_pixel_maps[i])
    agreement.append(r)
    print(f"{SHAPE_NAMES[i]:10s} | {base:11.3f} | {r:+.3f}")

print(f"\nсереднє по шести зображеннях: {np.nanmean(agreement):+.3f}")
print(f"розкид: від {min(agreement):+.3f} до {max(agreement):+.3f}")

**Три з шести — згода, три з шести — ні.** Мережа на всіх шести відповіла правильной упевнено, обидва методи відпрацювали без збоїв — а результат розходиться.Хто з них правий? Питання поставлене неправильно: еталонної карти не існує, бомережа нікуди не дивилась, вона перемножувала числа. Практичний висновок:**одна карта — це гіпотеза, а не результат**.

## 14 · ⚠️ ЗАМІР 6 · Перевірка на осудністьГоловний замір теми. Логіка проста й безжальна: **якщо карта пояснює те, щовивчила мережа, то в мережі, яка нічого не вивчила, карта має бути іншою.**Псуємо у два прийоми: спершу тільки голову (198 ваг), потім усе тіло.

In [ ]:
def randomize(model, what):
    """Копія моделі з випадковими вагами: 'head' — лише лінійний шар, 'all' — і згортки."""
    broken = copy.deepcopy(model)
    torch.manual_seed(1234)
    if what in ("head", "all"):
        nn.init.normal_(broken.fc.weight, std=0.1)
        nn.init.zeros_(broken.fc.bias)
    if what == "all":
        for layer in (broken.conv3, broken.conv2, broken.conv1):
            nn.init.kaiming_normal_(layer.weight, nonlinearity="relu")
            nn.init.zeros_(layer.bias)
    return broken


model_random_head = randomize(model, "head")
model_random_body = randomize(model, "all")

print(f"навчена мережа        : точність {test_accuracy:.3f}")
print(f"випадкова голова      : точність {accuracy(model_random_head, test_x, test_y):.3f}")
print(f"випадкове тіло й голова: точність {accuracy(model_random_body, test_x, test_y):.3f}")
print(f"вгадування навмання   : 0.167")
print("\nЗіпсовані мережі не «трохи гірші» — вони не знають нічого.")

Перш ніж дивитись на результат, потрібен **орієнтир**. Кореляція 0.2 — це багаточи мало? Щоб відповісти, навчимо другу мережу тієї самої форми з іншого випадковогопочатку. Вона теж розвʼязує задачу правильно, тож кореляція між картами двох**навчених** мереж — наша верхня планка.Це друге й останнє навчання в зошиті, ще близько 30 секунд.

In [ ]:
torch.manual_seed(1)
model_twin = Net()
started = time.perf_counter()
train(model_twin, train_x, train_y, seed=1)
print(f"друга мережа навчалась {time.perf_counter() - started:.1f} с, "
      f"точність {accuracy(model_twin, test_x, test_y):.3f}")

In [ ]:
print("зображення | інша навчена | вип. голова | вип. тіло")
head_scores, body_scores, twin_scores = [], [], []
for i in range(6):
    target_class = int(probe_pred[i])
    cam_head, _, _, _ = grad_cam(model_random_head, probe_batch[i], target_class)
    cam_body, _, _, _ = grad_cam(model_random_body, probe_batch[i], target_class)
    cam_twin, _, _, _ = grad_cam(model_twin, probe_batch[i], target_class)

    r_head = pearson(upsample(cams[i]), upsample(cam_head))
    r_body = pearson(upsample(cams[i]), upsample(cam_body))
    r_twin = pearson(upsample(cams[i]), upsample(cam_twin))
    head_scores.append(r_head)
    body_scores.append(r_body)
    twin_scores.append(r_twin)

    note = ""
    if cam_head.max() <= 1e-9:
        note = "  ← голова дала ПОРОЖНЮ карту"
    head_text = "   —   " if np.isnan(r_head) else f"{r_head:+.3f}"
    print(f"{SHAPE_NAMES[i]:10s} | {r_twin:+11.3f} | {head_text:>11s} | {r_body:+.3f}{note}")

print(f"\nСЕРЕДНЄ    | {np.nanmean(twin_scores):+11.3f} | "
      f"{np.nanmean(head_scores):+11.3f} | {np.nanmean(body_scores):+.3f}")

Кореляція — не єдиний спосіб порівняти дві карти. Перевіримо ще й так: скільки здванадцяти найяскравіших клітинок лишаються найяскравішими після псування ваг?Випадковий збіг дав би 12 ÷ 49 = 0.245.

In [ ]:
def top_overlap(a, b, k=12):
    """Частка спільних клітинок серед k найяскравіших у двох картах."""
    top_a = set(np.argsort(a.ravel())[-k:].tolist())
    top_b = set(np.argsort(b.ravel())[-k:].tolist())
    return len(top_a & top_b) / k


overlaps_head, overlaps_body = [], []
for i in range(6):
    target_class = int(probe_pred[i])
    overlaps_head.append(top_overlap(cams[i], grad_cam(model_random_head, probe_batch[i], target_class)[0]))
    overlaps_body.append(top_overlap(cams[i], grad_cam(model_random_body, probe_batch[i], target_class)[0]))

print(f"перекриття 12 найяскравіших клітинок, випадкова голова: {np.mean(overlaps_head):.3f}")
print(f"перекриття 12 найяскравіших клітинок, випадкове тіло  : {np.mean(overlaps_body):.3f}")
print(f"випадковий рівень (12 / 49)                           : {12 / 49:.3f}")

# і той самий тест для незалежного методу — заслону
drops_trained, _, _, _ = occlusion_map(model, probe_batch[0], int(probe_pred[0]), 7, 1)
drops_head, _, _, _ = occlusion_map(model_random_head, probe_batch[0], int(probe_pred[0]), 7, 1)
drops_body, _, _, _ = occlusion_map(model_random_body, probe_batch[0], int(probe_pred[0]), 7, 1)
print(f"\nзаслін під тим самим тестом («коло»): навчена ↔ вип. голова "
      f"{pearson(drops_trained, drops_head):+.3f}, "
      f"навчена ↔ вип. тіло {pearson(drops_trained, drops_body):+.3f}")

**Читаємо результат чесно, без помʼякшень.**Загалом Grad-CAM перевірку **проходить**: дві навчені мережі згодні між собою значносильніше, ніж навчена й зіпсована, а перекриття найяскравіших клітинок лежить нарівні випадкового.Але середнє ховає два неприємні факти:1. **на одному зображенні з шести карта не змінилась зовсім** — на кільці кореляція   з картою випадкової голови вища, ніж із картою другої навченої мережі;2. **на двох зображеннях метод узагалі не дав карти** — зважена сума виявилась   відʼємною скрізь, і `relu` стерла все.Звідки береться перше, ми вже вивели в розділі 9: при псуванні лише голови картиознак приходять із тіла, якого ми не чіпали, а випадкові ваги іноді випадковозберігають потрібні знаки.Висновок не «Grad-CAM працює» і не «Grad-CAM не працює», а такий: **відповідьзалежить від мережі, від зображення й від того, які саме ваги ти зіпсував — томуперевірку треба робити на своїй мережі, а не читати чужий висновок.**

## 15 · ЗАМІР 7 · Карта не пояснює причинуОстаннє. Візьмемо фігури в різних положеннях і порахуємо **центр ваги** карти —точку, навколо якої зосереджена її яскравість. Порівняємо з центром ваги самоїфігури. Дивимось не на красу карти, а на пару «правильна відповідь ↔ акуратна карта».

In [ ]:
def center_of_mass(matrix):
    """Точка, навколо якої зосереджена маса карти."""
    rows, cols = np.mgrid[0:matrix.shape[0], 0:matrix.shape[1]]
    total = matrix.sum() + 1e-12
    return float((matrix * rows).sum() / total), float((matrix * cols).sum() / total)


edge_rng = np.random.default_rng(11)
print("що на вході             | відповідь   | упевненість | центр карти ↔ центр фігури")
for kind, center in [(1, (5, 5)), (1, (14, 14)), (0, (5, 22)), (4, (22, 5))]:
    image = draw_shape(kind, edge_rng, center=center, radius=6)
    tensor = torch.from_numpy(image[None].astype(np.float32))
    with torch.no_grad():
        probs = torch.softmax(model(tensor.unsqueeze(0)), 1)[0]
    predicted = int(probs.argmax())

    cam, _, _, _ = grad_cam(model, tensor, predicted)
    cam_y, cam_x = center_of_mass(upsample(cam))
    image_y, image_x = center_of_mass(image)
    distance = float(np.hypot(cam_y - image_y, cam_x - image_x))

    mark = "✅" if predicted == kind else "❌"
    label = f"{SHAPE_NAMES[kind]} у ({center[0]:2d},{center[1]:2d})"
    print(f"{label:23s} | {SHAPE_NAMES[predicted]:11s} | {probs[predicted]:11.3f} | "
          f"{distance:.1f} px {mark}")

Знайди в таблиці рядок, де мережа **помилилась**, а відстань між центрами найменша.Це і є головний доказ розділу: карта бездоганна, відповідь хибна. І навпаки —є рядок із правильною відповіддю й помітно зсунутою картою.**Між правильністю відповіді й акуратністю карти немає ніякого звʼязку.**

## 16 · Підсумок зошита

In [ ]:
print("=" * 62)
print("СІМ ЗАМІРІВ ТЕМИ 17".center(62))
print("=" * 62)
print(f"1. ваги 1-го шару : найкращий збіг із Собелем "
      f"{best_edge_match(trained_kernels):.3f} проти {max(random_edge_scores):.3f} "
      f"у випадкового шуму")
print(f"2. активації      : ненульових "
      f"{layer_stats(a1[0])['nonzero']:.3f} → {layer_stats(a2[0])['nonzero']:.3f} → "
      f"{layer_stats(a3[0])['nonzero']:.3f}, мертвих карт "
      f"{layer_stats(a1[0])['dead']} → {layer_stats(a2[0])['dead']} → {layer_stats(a3[0])['dead']}")
print(f"3. Grad-CAM       : карта {cams[0].shape}, relu обнуляє {zeroed:.3f} клітинок")
print(f"4. заслін         : 484 прогони проти 2, максимальне падіння "
      f"{occlusion_map(model, probe_batch[0], int(probe_pred[0]), 7, 1)[0].max():.3f}")
print(f"5. згода методів  : {np.nanmean(agreement):+.3f} у середньому, "
      f"від {min(agreement):+.3f} до {max(agreement):+.3f}")
print(f"6. ОСУДНІСТЬ      : дві навчені {np.nanmean(twin_scores):+.3f} · "
      f"вип. голова {np.nanmean(head_scores):+.3f} · вип. тіло {np.nanmean(body_scores):+.3f}")
print(f"7. межі карти     : правильна відповідь може мати криву карту й навпаки")
print("=" * 62)
print(f"\nувесь зошит виконувався {time.perf_counter() - notebook_started:.0f} с")

## Завдання### 🟢 Рівень 1Побудуй Grad-CAM для зображення, на якому мережа **помилилась**. Знайди такезображення в перевірочній вибірці (`test_x`), побудуй карту для передбаченогокласу й окремо для правильного. Порівняй їх кореляцією.**Зроблено, якщо:** надруковано два числа — кореляція між двома картами й упевненістьмережі в хибній відповіді.### 🟡 Рівень 2Заслін заливає квадрат сірим (0.5). Це довільний вибір. Прогони `occlusion_map`із трьома заливками — `fill=0.0`, `fill=0.5` і `fill=1.0` — на одному зображенній порахуй кореляції між трьома отриманими картами.**Зроблено, якщо:** надрукована таблиця з трьох кореляцій і висновок одним реченням:наскільки результат заслону залежить від того, чим саме ми затуляємо.### 🔴 Рівень 3Прожени перевірку на осудність **повністю**: псуй шари по одному від виходу довходу (`fc`, потім `fc + conv3`, потім `fc + conv3 + conv2`, потім усе), і длякожного кроку порахуй середню кореляцію з картою навченої мережі по всіх шестизображеннях.**Зроблено, якщо:** надрукована таблиця з чотирьох рядків і сказано числом, на якомусаме кроці карта перестає бути схожою на початкову — і чи є такий крок узагалі.